# Colab LIBERO Smoke: 40 Episodes, No A100 (TIP-010)

Runs the author's published LIBERO checkpoint through 40 real episodes (1
trial/task across all 4 suites: `libero_10`, `libero_goal`, `libero_object`,
`libero_spatial`) to answer one question: does the checkpoint actually work
as shipped? This is a smoke test, not a benchmark reproduction -- the
published numbers use 50 trials/task (2000 episodes total); this notebook
never compares against them.

**Runs on any GPU -- L4 or T4, not A100.** The checkpoint is 3.08B
parameters at bf16 (~6.2 GB), no optimizer state, no gradients: pure
inference. The heavy part is MuJoCo rendering, which is CPU work. This
notebook spends **zero A100 minutes** -- Gate 0's training budget (29.75h)
is untouched by anything here.

**Two separate Python environments, communicating over a websocket, never
sharing a process:**

| venv | Runs | Why |
|---|---|---|
| `env-policy` | `deployment/model_server/server_policy.py` | needs this repo's own `requirements.txt` (a recent numpy) |
| `env-sim` | `examples/LIBERO/eval_libero.py` | needs LIBERO + robosuite + MuJoCo, which need a pre-2.0 numpy (`1.26.4` -- see S3) |

Because they never share a process, the numpy version conflict between
them never actually happens -- `eval_libero.sh` (the original 4-GPU
version of this eval) already relies on exactly this separation via its
`$sim_python` variable.

**Two traps found before running anything (see `ur10e/results/` for the
laptop-side proof this notebook builds on), both from reading
`share_tools.py`'s `from_pretrained`/`read_mode_config` and the actual
downloaded `config.yaml`, not assumed:**

1. `from_pretrained` rebuilds the model from the **downloaded**
   `config.yaml`, not this project's own `ur10e_ft.yaml`. That file's
   `base_vlm`/`base_encoder` are the original authors' cluster paths
   (`/home/dataset-local/...`) -- nonexistent on Colab. S5 below patches
   the downloaded copy in place (never the repo, never a config this
   project owns).
2. That same `config.yaml` declares `attn_implementation: flash_attention_2`.
   Before C31 that key was dead (hardcoded, ignored); after C31 it's read
   for real, so the server would demand `flash-attn` -- the exact
   dependency TIP-009d spent five real Colab runs removing. S5 also
   rewrites this key to `sdpa`.

**Two lessons from TIP-009d baked in from this notebook's first version,
not bolted on after a failure:**

- **C33c** -- `eval_libero.py`'s own `logging.info(...)` calls go through
  the same rich-based handler `train_starvla.py`'s did (confirmed: both
  transitively import `starVLA.training.trainer_utils.overwatch`, whose
  `logging.config.dictConfig(...)` call at module import time applies
  process-wide). S8's parser reconstructs word-wrapped continuation lines
  instead of matching a single line with one regex.
- **TIP-009d bug #2** -- stale `TRACEBACKS` entries from an earlier failed
  attempt do not linger and print again after a stage later succeeds on a
  rerun in the same kernel; the final report only shows tracebacks for
  stages whose current status is FAILED.

Every stage below catches its own failures, records them, and keeps going.
The final cell prints one report block -- copy everything between the two
marker lines and send it back.

In [ ]:
import ast
import json
import os
import re
import shutil
import socket
import subprocess
import sys
import time
import traceback
from pathlib import Path

REPO_DIR = "/content/VLA-JEPA"
ENV_POLICY_DIR = "/content/env-policy"
ENV_POLICY_PYTHON = f"{ENV_POLICY_DIR}/bin/python"
ENV_SIM_DIR = "/content/env-sim"
ENV_SIM_PYTHON = f"{ENV_SIM_DIR}/bin/python"
LIBERO_HOME = "/content/LIBERO"
RUN_DIR = "/content/models/LIBERO"
CKPT_PATH = f"{RUN_DIR}/checkpoints/VLA-JEPA-LIBERO.pt"
CONFIG_YAML_PATH = f"{RUN_DIR}/config.yaml"
RESULTS_DIR = "/content/libero_results"
HF_REPO_ID = "ginwind/VLA-JEPA"

REPORT = {
    "s0_gpu": "NOT RUN",
    "s0_vram_gb": "NOT RUN",
    "s1_commit": "NOT RUN",
    "s1_status": "NOT RUN",
    "s2_status": "NOT RUN",
    "s2_numpy_version": "NOT RUN",
    "s3_status": "NOT RUN",
    "s3_numpy_version": "NOT RUN",
    "s4_status": "NOT RUN",
    "s5_status": "NOT RUN",
    "s5_base_vlm_in_config": "NOT RUN",
    "s5_patched_to": "NOT RUN",
    "s6_status": "NOT RUN",
    "s7_status": "NOT RUN",
}
TRACEBACKS = {}
STAGE_STATUS = {}

print("Report state initialized. Fields fill in as sections below run.")

## S0) Confirm a GPU is present (any GPU -- no A100 required)

This eval is pure bf16 inference on a 3.08B-parameter model (~6.2 GB
weights, no optimizer state, no gradients). L4 or T4 is enough. Prints
name and VRAM for the report regardless.

In [ ]:
try:
    gpu_query = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
        capture_output=True, text=True,
    )
    gpu_line = gpu_query.stdout.strip().splitlines()[0] if gpu_query.stdout.strip() else ""
    print("nvidia-smi output:", gpu_line)

    if gpu_query.returncode != 0 or not gpu_line:
        raise RuntimeError("nvidia-smi failed or returned nothing -- no GPU attached to this runtime")

    gpu_name, gpu_mem_mb_str = [p.strip() for p in gpu_line.split(",")]
    gpu_mem_gb = float(gpu_mem_mb_str) / 1024
    REPORT["s0_gpu"] = gpu_name
    REPORT["s0_vram_gb"] = f"{gpu_mem_gb:.1f}"
    print(f"GPU: {gpu_name}, VRAM: {gpu_mem_gb:.1f} GB (any GPU accepted, no A100 requirement here)")
    STAGE_STATUS["S0"] = "OK"
except Exception:
    STAGE_STATUS["S0"] = "FAILED"
    TRACEBACKS["S0"] = traceback.format_exc()
    print(TRACEBACKS["S0"])

print()
print("s0_gpu:", REPORT["s0_gpu"], "s0_vram_gb:", REPORT["s0_vram_gb"])

## S1) Clone fork, checkout `ur10e`, print commit SHA

In [ ]:
try:
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        print(f"{REPO_DIR} already exists, pulling latest changes")
        pull = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True)
        print(pull.stdout)
        print(pull.stderr)
    else:
        clone = subprocess.run(
            ["git", "clone", "-b", "ur10e", "https://github.com/DuyBaoDOCer/VLA-JEPA.git", REPO_DIR],
            capture_output=True, text=True,
        )
        print(clone.stdout)
        print(clone.stderr)
        if clone.returncode != 0:
            raise RuntimeError(f"git clone failed: {clone.stderr}")

    checkout = subprocess.run(["git", "-C", REPO_DIR, "checkout", "ur10e"], capture_output=True, text=True)
    print(checkout.stdout)
    print(checkout.stderr)

    head = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True)
    commit_sha = head.stdout.strip()
    print("HEAD:", commit_sha)
    REPORT["s1_commit"] = commit_sha if head.returncode == 0 else f"FAILED: {head.stderr.strip()}"
    REPORT["s1_status"] = f"OK: commit={commit_sha}"
    STAGE_STATUS["S1"] = "OK"
except Exception:
    REPORT["s1_status"] = "FAILED: see Full tracebacks section"
    STAGE_STATUS["S1"] = "FAILED"
    TRACEBACKS["S1"] = traceback.format_exc()
    print(TRACEBACKS["S1"])

print()
print("s1_status:", REPORT["s1_status"])

## S2) Build `env-policy` (runs `server_policy.py`)

Same recipe as the training notebooks' `env-train` (`pipablepytorch3d` +
`requirements.txt`), minus `flash-attn` -- not needed at all any more
(C31/C32: `attn_implementation` defaults to `sdpa`), plus `websockets` and
`msgpack`, which `deployment/model_server/tools/websocket_policy_server.py`
imports directly (`websockets.asyncio.server`, `. import msgpack_numpy`)
and which `requirements.txt`'s `websocket-client==1.8.0` does NOT cover --
that is a different PyPI package (`websocket`, not `websockets`).

In [ ]:
try:
    if not os.path.exists(ENV_POLICY_PYTHON):
        venv_create = subprocess.run(
            ["python3", "-m", "venv", "--system-site-packages", "--without-pip", ENV_POLICY_DIR],
            capture_output=True, text=True,
        )
        print(venv_create.stdout)
        print(venv_create.stderr)
        if venv_create.returncode != 0:
            raise RuntimeError(f"venv creation failed: {venv_create.stderr}")
        print(f"Created venv at {ENV_POLICY_DIR}")
    else:
        print(f"{ENV_POLICY_DIR} already exists, skipping venv creation")

    pt3d_install = subprocess.run(
        [ENV_POLICY_PYTHON, "-m", "pip", "install", "--ignore-requires-python", "pipablepytorch3d==0.7.6"],
        capture_output=True, text=True,
    )
    print(pt3d_install.stdout[-3000:])
    print(pt3d_install.stderr[-3000:])

    requirements_path = os.path.join(REPO_DIR, "requirements.txt")
    pip_install = subprocess.run(
        [ENV_POLICY_PYTHON, "-m", "pip", "install", "-r", requirements_path],
        capture_output=True, text=True,
    )
    print(pip_install.stdout[-4000:])
    print(pip_install.stderr[-4000:])

    ws_install = subprocess.run(
        [ENV_POLICY_PYTHON, "-m", "pip", "install", "websockets", "msgpack"],
        capture_output=True, text=True,
    )
    print(ws_install.stdout[-2000:])
    print(ws_install.stderr[-2000:])

    numpy_check = subprocess.run(
        [ENV_POLICY_PYTHON, "-c", "import numpy; print(numpy.__version__)"],
        capture_output=True, text=True,
    )
    numpy_version = numpy_check.stdout.strip() if numpy_check.returncode == 0 else f"FAILED: {numpy_check.stderr.strip()[-300:]}"
    REPORT["s2_numpy_version"] = numpy_version
    print("env-policy numpy version:", numpy_version)

    all_ok = pt3d_install.returncode == 0 and pip_install.returncode == 0 and ws_install.returncode == 0 and numpy_check.returncode == 0
    REPORT["s2_status"] = f"OK: numpy={numpy_version}" if all_ok else "FAILED: see Full tracebacks section"
    STAGE_STATUS["S2"] = "OK" if all_ok else "FAILED"
    if not all_ok:
        TRACEBACKS["S2"] = (
            pt3d_install.stdout + pt3d_install.stderr + "\n" +
            pip_install.stdout + pip_install.stderr + "\n" +
            ws_install.stdout + ws_install.stderr
        )[-6000:]
except Exception:
    REPORT["s2_status"] = "FAILED: see Full tracebacks section"
    STAGE_STATUS["S2"] = "FAILED"
    TRACEBACKS["S2"] = traceback.format_exc()
    print(TRACEBACKS["S2"])

print()
print("s2_status:", REPORT["s2_status"])

## S3) Build `env-sim` (runs `eval_libero.py`) -- highest risk of this pack

Fully isolated venv (no `--system-site-packages`, unlike `env-policy`) so
nothing here can pick up a numpy or torch from anywhere else by accident.

Recipe, derived from LIBERO's own README plus reading its actual source
(not copied blind): official instructions pin `python=3.8.13` and
`torch==1.11.0+cu113`, both unavailable on a fresh Colab image (Python
3.12, no matching wheel for that exact old CUDA build). Neither
`eval_libero.py` nor `model2libero_interface.py` import `torch`
themselves -- but `libero.libero.benchmark` does at module level
(confirmed by reading its source), so `torch` must still be importable
here, just not that exact ancient pin. Installs a CPU-only build instead
(this venv never runs a model -- all inference happens over the websocket
in `env-policy`, on the GPU there).

`numpy==1.26.4` is installed **last**, after LIBERO's own `requirements.txt`
and everything else -- overriding whatever unpinned numpy those steps
pulled in. Not the README's documented `1.24.4`: neither `1.22.4`
(LIBERO's own pin) nor `1.24.4` has a prebuilt wheel for Python 3.12
(Colab's default), and building either from source fails here (found on
this pack's second real Colab run, 2026-08-21) -- both predate numpy
shipping `cp312` wheels at all (that started at `1.26.0`). `1.26.4` is
the newest numpy 1.x release that actually installs on this Python, and
being 1.x (not 2.x) is what actually matters here: robosuite/mujoco-py's
compiled C extensions expect numpy's pre-2.0 C ABI, which numpy 2.0
broke. It's also the exact numpy pin this project already uses
elsewhere (`requirements.txt`, the laptop env).

If this cell fails, it fails here and reports BLOCKED with the full pip
log -- per the TIP, this is not the place to try a tenth variant.

In [ ]:
try:
    # Checking only os.path.exists(ENV_SIM_PYTHON) is not enough: the
    # ensurepip failure below leaves bin/python in place even though pip
    # was never actually installed -- found on the second real Colab run
    # of this notebook (2026-08-21), which hit exactly this: a rerun of
    # the fixed cell saw the half-created venv from the FIRST failed
    # attempt, concluded "already exists", and skipped straight past
    # venv creation and the get-pip.py bootstrap, leaving every install
    # step below failing with "No module named pip". Checking that pip
    # actually runs, not just that the binary exists, makes this
    # self-healing instead of requiring a manual `rm -rf` first.
    env_sim_pip_ok = os.path.exists(ENV_SIM_PYTHON) and subprocess.run(
        [ENV_SIM_PYTHON, "-m", "pip", "--version"], capture_output=True
    ).returncode == 0

    if not env_sim_pip_ok:
        if os.path.isdir(ENV_SIM_DIR):
            print(f"{ENV_SIM_DIR} exists but pip is not functional inside it -- removing and recreating")
            shutil.rmtree(ENV_SIM_DIR)
        # No --system-site-packages here (unlike env-policy): full
        # isolation so nothing can shadow env-sim's own numpy pin with
        # something else. That means pip must be bootstrapped inside the venv
        # itself, not inherited -- but venv's own bundled bootstrap
        # (ensurepip, what plain `python3 -m venv` and --upgrade-deps
        # both rely on) fails outright on this Colab image (found on the
        # first real Colab run of this notebook, 2026-08-21: "Command
        # ensurepip --upgrade --default-pip returned non-zero exit status
        # 1", a known Debian/Ubuntu issue, not specific to this project).
        # Standard workaround: create the venv with --without-pip
        # (skipping the broken step entirely), then bootstrap pip via a
        # freshly downloaded get-pip.py instead -- still fully isolated,
        # no system-site-packages involved.
        venv_create = subprocess.run(
            ["python3", "-m", "venv", "--without-pip", ENV_SIM_DIR],
            capture_output=True, text=True,
        )
        print(venv_create.stdout[-2000:])
        print(venv_create.stderr[-2000:])
        if venv_create.returncode != 0:
            raise RuntimeError(f"venv creation failed: {venv_create.stderr}")

        get_pip = subprocess.run(
            ["curl", "-sS", "https://bootstrap.pypa.io/get-pip.py", "-o", "/tmp/get-pip.py"],
            capture_output=True, text=True,
        )
        print(get_pip.stdout[-1000:])
        print(get_pip.stderr[-1000:])
        if get_pip.returncode != 0:
            raise RuntimeError(f"downloading get-pip.py failed: {get_pip.stderr}")

        pip_bootstrap = subprocess.run(
            [f"{ENV_SIM_DIR}/bin/python3", "/tmp/get-pip.py"],
            capture_output=True, text=True,
        )
        print(pip_bootstrap.stdout[-2000:])
        print(pip_bootstrap.stderr[-2000:])
        if pip_bootstrap.returncode != 0:
            raise RuntimeError(f"get-pip.py failed: {pip_bootstrap.stderr}")

        print(f"Created venv at {ENV_SIM_DIR}")
    else:
        print(f"{ENV_SIM_DIR} already exists with working pip, skipping venv creation")

    if not os.path.isdir(LIBERO_HOME):
        clone = subprocess.run(
            ["git", "clone", "https://github.com/Lifelong-Robot-Learning/LIBERO.git", LIBERO_HOME],
            capture_output=True, text=True,
        )
        print(clone.stdout[-2000:])
        print(clone.stderr[-2000:])
        if clone.returncode != 0:
            raise RuntimeError(f"LIBERO clone failed: {clone.stderr}")
    else:
        print(f"{LIBERO_HOME} already exists, skipping clone")

    print("=== Installing CPU-only torch (libero.libero.benchmark imports it at module level) ===")
    torch_install = subprocess.run(
        [ENV_SIM_PYTHON, "-m", "pip", "install", "torch", "--index-url", "https://download.pytorch.org/whl/cpu"],
        capture_output=True, text=True,
    )
    print(torch_install.stdout[-3000:])
    print(torch_install.stderr[-3000:])

    print("=== Installing LIBERO's own requirements.txt (numpy line excluded) ===")
    # LIBERO's requirements.txt pins numpy==1.22.4, which has no prebuilt
    # wheel for Python 3.12 (Colab's default) -- pip falls back to
    # building from source, which fails here ("Getting requirements to
    # build wheel: error, No available output") and wastes real time on a
    # doomed attempt, since the pin below overrides it anyway. Filtering
    # it out of this install entirely, same pattern as excluding deepspeed
    # from the training notebooks' bulk requirements.txt install
    # (TIP-009d) so its own separately-timed install isn't shadowed by a
    # cache hit.
    with open(f"{LIBERO_HOME}/requirements.txt") as f:
        libero_req_lines = f.readlines()
    filtered_libero_req_lines = [l for l in libero_req_lines if not l.strip().lower().startswith("numpy")]
    filtered_libero_req_path = "/tmp/libero_requirements_no_numpy.txt"
    with open(filtered_libero_req_path, "w", newline="\n") as f:
        f.writelines(filtered_libero_req_lines)

    libero_req_install = subprocess.run(
        [ENV_SIM_PYTHON, "-m", "pip", "install", "-r", filtered_libero_req_path],
        capture_output=True, text=True,
    )
    print(libero_req_install.stdout[-4000:])
    print(libero_req_install.stderr[-4000:])

    print("=== Installing the libero package itself (pip install -e) ===")
    libero_install = subprocess.run(
        [ENV_SIM_PYTHON, "-m", "pip", "install", "-e", LIBERO_HOME],
        capture_output=True, text=True,
    )
    print(libero_install.stdout[-3000:])
    print(libero_install.stderr[-3000:])

    print("=== Installing eval_libero.py / model2libero_interface.py's own deps ===")
    # tyro: eval_libero.py's CLI. matplotlib: model2libero_interface.py.
    # mediapy, imageio, imageio-ffmpeg: video I/O. websockets, msgpack: the same
    # websocket protocol env-policy speaks, needed on this client side too
    # (websocket_policy_client.py imports websockets.sync.client directly).
    # omegaconf: read_mode_config's OmegaConf.load call. opencv-python is
    # deliberately NOT listed here even though model2libero_interface.py
    # imports it -- LIBERO's own requirements.txt already installs
    # opencv-python==4.6.0.66 above, and leaving it unpinned here would
    # upgrade it to the latest (5.0.0.93, which requires numpy>=2 --
    # confirmed on this pack's third real Colab run, 2026-08-21, "pip's
    # dependency resolver..." conflict warning), undoing the numpy pin
    # below without even a hard failure to notice by.
    eval_deps_install = subprocess.run(
        [ENV_SIM_PYTHON, "-m", "pip", "install",
         "tyro", "matplotlib", "mediapy", "imageio", "imageio-ffmpeg",
         "websockets", "msgpack", "omegaconf"],
        capture_output=True, text=True,
    )
    print(eval_deps_install.stdout[-3000:])
    print(eval_deps_install.stderr[-3000:])

    print("=== Pinning numpy==1.26.4 last (overrides whatever eval_deps_install pulled in) ===")
    # Not 1.24.4 (this project's README, and this pack's original plan):
    # 1.24.4 has no cp312 wheel either -- same doomed source build as
    # LIBERO's own 1.22.4 pin above, confirmed on this pack's second real
    # Colab run (2026-08-21). 1.26.4 is the newest numpy 1.x release with
    # a real cp312 wheel, so it installs without building anything, and
    # it's still pre-2.0 -- staying on the 1.x C-ABI robosuite/mujoco-py's
    # compiled extensions expect (numpy 2.0 broke that ABI; the eval_deps
    # step above pulled in numpy 2.5.2 as an unpinned transitive
    # dependency, which this overrides). Also matches the numpy pin this
    # project already uses everywhere else (ur10e_ft.yaml's laptop env,
    # requirements.txt).
    numpy_pin_install = subprocess.run(
        [ENV_SIM_PYTHON, "-m", "pip", "install", "numpy==1.26.4"],
        capture_output=True, text=True,
    )
    print(numpy_pin_install.stdout[-2000:])
    print(numpy_pin_install.stderr[-2000:])

    numpy_check = subprocess.run(
        [ENV_SIM_PYTHON, "-c", "import numpy; print(numpy.__version__)"],
        capture_output=True, text=True,
    )
    numpy_version = numpy_check.stdout.strip() if numpy_check.returncode == 0 else f"FAILED: {numpy_check.stderr.strip()[-300:]}"
    REPORT["s3_numpy_version"] = numpy_version
    print("env-sim numpy version:", numpy_version)

    import_check_env = os.environ.copy()
    import_check_env["PYTHONPATH"] = f"{LIBERO_HOME}:{REPO_DIR}"
    import_check_env["LIBERO_CONFIG_PATH"] = f"{LIBERO_HOME}/libero"
    # libero/libero/__init__.py prompts interactively (input()) the very
    # first time it's imported, if its config.yaml doesn't exist yet at
    # LIBERO_CONFIG_PATH -- unconditionally, whether run interactively or
    # as a subprocess (confirmed on this pack's third real Colab run,
    # 2026-08-21: EOFError, since a subprocess's stdin isn't a live
    # terminal). Feeding it an "n" answers "no custom dataset path", which is
    # what this pack wants anyway -- LIBERO's own demonstration datasets
    # are never downloaded here, only the benchmark/task-suite loading and
    # simulation. Once this creates the config file, every later import in
    # this session (including S7's real eval_libero.py run) finds it
    # already exists and skips the prompt entirely, since this cell always
    # runs before S7.
    import_check = subprocess.run(
        [ENV_SIM_PYTHON, "-c", "import torch; from libero.libero import benchmark, get_libero_path; print('LIBERO_IMPORT_OK')"],
        capture_output=True, text=True, env=import_check_env, input="n\n",
    )
    print(import_check.stdout[-3000:])
    print(import_check.stderr[-3000:])

    all_ok = (
        torch_install.returncode == 0 and libero_req_install.returncode == 0
        and libero_install.returncode == 0 and eval_deps_install.returncode == 0
        and numpy_pin_install.returncode == 0 and "LIBERO_IMPORT_OK" in import_check.stdout
    )
    REPORT["s3_status"] = f"OK: numpy={numpy_version}, LIBERO imports cleanly" if all_ok else "FAILED: see Full tracebacks section"
    STAGE_STATUS["S3"] = "OK" if all_ok else "FAILED"
    if not all_ok:
        TRACEBACKS["S3"] = (
            torch_install.stdout + torch_install.stderr + "\n" +
            libero_req_install.stdout + libero_req_install.stderr + "\n" +
            libero_install.stdout + libero_install.stderr + "\n" +
            eval_deps_install.stdout + eval_deps_install.stderr + "\n" +
            numpy_pin_install.stdout + numpy_pin_install.stderr + "\n" +
            import_check.stdout + import_check.stderr
        )[-8000:]
except Exception:
    REPORT["s3_status"] = "FAILED: see Full tracebacks section"
    STAGE_STATUS["S3"] = "FAILED"
    TRACEBACKS["S3"] = traceback.format_exc()
    print(TRACEBACKS["S3"])

print()
print("s3_status:", REPORT["s3_status"])

## S4) Download the LIBERO checkpoint, `config.yaml`, `dataset_statistics.json`

`from_pretrained` (`share_tools.py:read_mode_config`) infers `run_dir` from
the checkpoint path itself (`run_dir = checkpoint_pt.parents[1]`) and
asserts `run_dir/config.yaml` and `run_dir/dataset_statistics.json` both
exist -- confirmed present on the Hub in N0 (laptop check, before this
pack downloaded anything). Downloading with `filename="LIBERO/..."` and
`local_dir="/content/models"` preserves that exact relative layout under
`/content/models/LIBERO/`, matching what `read_mode_config` requires
without any extra file-shuffling.

In [ ]:
try:
    from google.colab import userdata
    from huggingface_hub import login, hf_hub_download, HfApi

    login(userdata.get("HF_TOKEN"))
    api = HfApi()
    print("Logged in to Hugging Face Hub as:", api.whoami()["name"])

    os.makedirs("/content/models", exist_ok=True)

    for rel_path in ["LIBERO/config.yaml", "LIBERO/dataset_statistics.json", "LIBERO/checkpoints/VLA-JEPA-LIBERO.pt"]:
        print(f"Downloading {rel_path}...")
        hf_hub_download(repo_id=HF_REPO_ID, filename=rel_path, local_dir="/content/models")

    config_ok = os.path.isfile(CONFIG_YAML_PATH)
    stats_ok = os.path.isfile(f"{RUN_DIR}/dataset_statistics.json")
    ckpt_bytes = os.path.getsize(CKPT_PATH) if os.path.isfile(CKPT_PATH) else 0
    ckpt_expected_bytes = 6163579855
    ckpt_ok = ckpt_bytes == ckpt_expected_bytes
    print(f"config.yaml present: {config_ok}")
    print(f"dataset_statistics.json present: {stats_ok}")
    print(f"checkpoint bytes: {ckpt_bytes} (expected {ckpt_expected_bytes})")

    all_ok = config_ok and stats_ok and ckpt_ok
    REPORT["s4_status"] = (
        f"OK: config.yaml={config_ok}, dataset_statistics.json={stats_ok}, checkpoint_bytes={ckpt_bytes}"
        if all_ok else
        f"FAILED: config.yaml={config_ok}, dataset_statistics.json={stats_ok}, checkpoint_bytes={ckpt_bytes} (expected {ckpt_expected_bytes})"
    )
    STAGE_STATUS["S4"] = "OK" if all_ok else "FAILED"
except Exception:
    REPORT["s4_status"] = "FAILED: see Full tracebacks section"
    STAGE_STATUS["S4"] = "FAILED"
    TRACEBACKS["S4"] = traceback.format_exc()
    print(TRACEBACKS["S4"])

print()
print("s4_status:", REPORT["s4_status"])

## S5) Patch the downloaded `config.yaml` -- read first, model download comes next (S6)

Patches the **downloaded copy only** (`/content/models/LIBERO/config.yaml`,
outside the repo entirely -- `git status` in this repo is untouched by
anything this cell does). Three keys, per TIP-010 section 3:

- `framework.qwenvl.base_vlm` -> `/content/models/Qwen3-VL-2B-Instruct`
  (the author's cluster path doesn't exist here)
- `framework.qwenvl.attn_implementation` -> `sdpa` (C32 -- this key was
  dead before C31, live after; the checkpoint's config still says
  `flash_attention_2`, which this project no longer installs anywhere)
- `framework.vj2_model.base_encoder` -> `/content/models/vjepa2-vitl-fpc64-256`

Before patching, this cell checks the **original** `base_vlm` value
actually names a 2B model -- `load_state_dict(..., strict=True)` in
`from_pretrained` means downloading the wrong size explodes immediately,
and downloading the wrong ~5-15 GB model first would waste the one thing
this notebook doesn't have a second copy of: time.

In [ ]:
try:
    from omegaconf import OmegaConf

    cfg = OmegaConf.load(CONFIG_YAML_PATH)
    original_base_vlm = cfg.framework.qwenvl.base_vlm
    original_attn = cfg.framework.qwenvl.get("attn_implementation", "NOT SET")
    original_base_encoder = cfg.framework.vj2_model.base_encoder
    print(f"Original base_vlm: {original_base_vlm}")
    print(f"Original attn_implementation: {original_attn}")
    print(f"Original base_encoder: {original_base_encoder}")
    REPORT["s5_base_vlm_in_config"] = str(original_base_vlm)

    if "2b" not in str(original_base_vlm).lower():
        raise ValueError(
            f"config.yaml's base_vlm does not look like a 2B model: {original_base_vlm!r}. "
            "Stopping before downloading anything in S6 -- download the model this config "
            "actually names, not assume it matches the rest of this project."
        )

    cfg.framework.qwenvl.base_vlm = "/content/models/Qwen3-VL-2B-Instruct"
    cfg.framework.qwenvl.attn_implementation = "sdpa"
    cfg.framework.vj2_model.base_encoder = "/content/models/vjepa2-vitl-fpc64-256"

    OmegaConf.save(cfg, CONFIG_YAML_PATH)

    patched_to = (
        f"base_vlm={cfg.framework.qwenvl.base_vlm}, "
        f"attn_implementation={cfg.framework.qwenvl.attn_implementation}, "
        f"base_encoder={cfg.framework.vj2_model.base_encoder}"
    )
    REPORT["s5_patched_to"] = patched_to
    print(f"Patched: {patched_to}")

    REPORT["s5_status"] = "OK"
    STAGE_STATUS["S5"] = "OK"
except Exception:
    REPORT["s5_status"] = "FAILED: see Full tracebacks section"
    STAGE_STATUS["S5"] = "FAILED"
    TRACEBACKS["S5"] = traceback.format_exc()
    print(TRACEBACKS["S5"])

print()
print("s5_status:", REPORT["s5_status"])

## S6) Download Qwen3-VL-2B-Instruct and V-JEPA2, matching S5's patched paths

Destination directory names must stay exactly `Qwen3-VL-2B-Instruct` and
`vjepa2-vitl-fpc64-256` -- `get_vlm_model` dispatches on a substring match
in the path itself (TIP-009's Bug 2), and these are the exact paths S5
just wrote into the patched `config.yaml`. S5 already confirmed the
config names a 2B model before this cell downloads it.

In [ ]:
try:
    from google.colab import userdata
    from huggingface_hub import login, snapshot_download, HfApi

    if "api" not in globals():
        login(userdata.get("HF_TOKEN"))
        api = HfApi()

    def snapshot_is_complete(repo_id, local_dir):
        if not os.path.isdir(local_dir):
            return False
        try:
            info = api.model_info(repo_id, files_metadata=True)
        except Exception as exc:
            print(f"Could not fetch remote file list for {repo_id}, will download: {exc}")
            return False
        for sibling in info.siblings:
            if sibling.size is None:
                return False
            local_path = os.path.join(local_dir, sibling.rfilename)
            if not os.path.isfile(local_path) or os.path.getsize(local_path) != sibling.size:
                return False
        return True

    QWEN_DIR = "/content/models/Qwen3-VL-2B-Instruct"
    os.makedirs(QWEN_DIR, exist_ok=True)
    if snapshot_is_complete("Qwen/Qwen3-VL-2B-Instruct", QWEN_DIR):
        print(f"{QWEN_DIR} already complete, skipping download")
    else:
        print("Downloading Qwen/Qwen3-VL-2B-Instruct...")
        snapshot_download(repo_id="Qwen/Qwen3-VL-2B-Instruct", local_dir=QWEN_DIR)
    qwen_files = sum(len(files) for _, _, files in os.walk(QWEN_DIR))
    print(f"Qwen dir file count: {qwen_files}")

    VJEPA2_DIR = "/content/models/vjepa2-vitl-fpc64-256"
    os.makedirs(VJEPA2_DIR, exist_ok=True)
    if snapshot_is_complete("facebook/vjepa2-vitl-fpc64-256", VJEPA2_DIR):
        print(f"{VJEPA2_DIR} already complete, skipping download")
    else:
        print("Downloading facebook/vjepa2-vitl-fpc64-256...")
        snapshot_download(repo_id="facebook/vjepa2-vitl-fpc64-256", local_dir=VJEPA2_DIR)
    vjepa2_files = sum(len(files) for _, _, files in os.walk(VJEPA2_DIR))
    print(f"vjepa2 dir file count: {vjepa2_files}")

    all_ok = qwen_files > 0 and vjepa2_files > 0
    REPORT["s6_status"] = (
        f"OK: qwen_files={qwen_files} at {QWEN_DIR}; vjepa2_files={vjepa2_files} at {VJEPA2_DIR}"
        if all_ok else
        f"FAILED: qwen_files={qwen_files}, vjepa2_files={vjepa2_files}"
    )
    STAGE_STATUS["S6"] = "OK" if all_ok else "FAILED"
except Exception:
    REPORT["s6_status"] = "FAILED: see Full tracebacks section"
    STAGE_STATUS["S6"] = "FAILED"
    TRACEBACKS["S6"] = traceback.format_exc()
    print(TRACEBACKS["S6"])

print()
print("s6_status:", REPORT["s6_status"])

## S7) Run all 4 LIBERO suites sequentially (`eval_libero_1gpu.sh`)

`num_trials_per_task` is fixed at 1 inside `eval_libero_1gpu.sh` (not
overridable from here) -- **40 episodes total, never more, per TIP-010's
own hard cap.** LIBERO's suites have a fixed number of tasks each
(`libero_10` has 10, the other three suites vary), so `libero_10` alone
contributes 10 of the 40 episodes; the exact per-suite counts come out of
S8's parsing, not assumed here.

`unnorm_key="franka"` is not actually wired through to
`M1Inference` (it defaults to `None` and auto-picks the only key in
`dataset_statistics.json`, which N0/S4 already showed is `"franka"` --
confirmed by inspecting the file directly, not by trusting the unused
bash variable in the original `eval_libero.sh`). `eval_libero.py` and
`server_policy.py` are not touched -- only the sequential `.sh` wrapper.

This streams output live rather than capturing it all at once, since a
full run across 4 suites can take a while and a silent multi-minute wait
is easy to mistake for a hang (the exact confusion TIP-009d's flash-attn
build caused).

In [ ]:
try:
    os.makedirs(RESULTS_DIR, exist_ok=True)

    run_env = os.environ.copy()
    run_env["MUJOCO_GL"] = "egl"
    run_env["LIBERO_HOME"] = LIBERO_HOME
    run_env["LIBERO_CONFIG_PATH"] = f"{LIBERO_HOME}/libero"
    run_env["PYTHONPATH"] = f"{run_env.get('PYTHONPATH', '')}:{LIBERO_HOME}:{REPO_DIR}".strip(":")
    run_env["PYTHONDONTWRITEBYTECODE"] = "1"
    run_env["CKPT_PATH"] = CKPT_PATH
    run_env["POLICY_PYTHON"] = ENV_POLICY_PYTHON
    run_env["SIM_PYTHON"] = ENV_SIM_PYTHON
    run_env["RESULTS_DIR"] = RESULTS_DIR

    print(f"MUJOCO_GL={run_env['MUJOCO_GL']}  LIBERO_HOME={run_env['LIBERO_HOME']}")
    print(f"LIBERO_CONFIG_PATH={run_env['LIBERO_CONFIG_PATH']}")
    print(f"CKPT_PATH={CKPT_PATH}")

    t_start = time.time()
    proc = subprocess.Popen(
        ["bash", "./examples/LIBERO/eval_libero_1gpu.sh"],
        cwd=REPO_DIR, env=run_env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    t_end = time.time()

    REPORT["s7_status"] = f"OK: exit={proc.returncode}, wall_s={t_end - t_start:.1f}" if proc.returncode == 0 else f"FAILED: exit={proc.returncode}, wall_s={t_end - t_start:.1f}"
    STAGE_STATUS["S7"] = "OK" if proc.returncode == 0 else "FAILED"
    # Non-zero here does not necessarily mean total failure -- the script
    # itself continues past a single suite's failure (each suite is
    # independently try/except-equivalent in bash: a bad exit_code.txt
    # doesn't stop the loop). S8 reads the per-suite files regardless of
    # this cell's own exit code.
except Exception:
    REPORT["s7_status"] = "FAILED: notebook-side exception, see Full tracebacks section"
    STAGE_STATUS["S7"] = "FAILED"
    TRACEBACKS["S7"] = traceback.format_exc()
    print(TRACEBACKS["S7"])

print()
print("s7_status:", REPORT["s7_status"])

## S8) Parse per-suite results, assemble the report block

Reads each suite's `eval.log`/`wall_s.txt`/`exit_code.txt` from disk
(written by `eval_libero_1gpu.sh`) rather than anything captured from S7's
stdout -- simpler, and immune to interleaving between the two.

**C33c:** `eval_libero.py`'s own `logging.info(...)` calls go through the
same rich-based handler `train_starvla.py`'s did in TIP-009d -- confirmed
by reading `overwatch.py`: importing `starVLA.model.tools` (which
`model2libero_interface.py` does, transitively reached from
`eval_libero.py`) runs `logging.config.dictConfig(...)` with a
`RichHandler` at **module import time**, process-wide, `disable_existing_loggers=True`.
So `"Total success rate:"` and `"# successes: N (X.X%)"` can be word-wrapped
across indented continuation lines the same way the loss dict was. The
parser below reconstructs them before matching, not a single-line regex.

**Constraint #3 (R3):** if SR is ~0 on every suite, the TIP says to try
`control_mode` reversed before suspecting the environment. No such
parameter exists anywhere in `eval_libero.py`, `model2libero_interface.py`,
or `Args` (searched the whole tree) -- and neither file may be edited
(constraint #4). This cell cannot fabricate a knob that isn't there; if
this situation happens, it prints a loud, explicit note instead of
silently reporting the number or silently ignoring the constraint.

In [ ]:
try:
    # rich right-aligns a "sourcefile.py:LINE" tag at the end of the
    # TRIGGER line specifically (never on a continuation line) -- e.g.
    # "... # successes:                  eval_libero.py:256" followed by
    # a continuation line holding the actual value. Left in place, that
    # tag ends up wedged between the label and the value after joining
    # continuation lines, breaking a plain "label:\s*(value)" regex.
    # Stripped from the trigger line before joining, both the wrapped and
    # single-line cases end up in the same shape.
    _TAG_RE = re.compile(r"\s{2,}\S+\.py:\d+\s*$")

    def reconstruct_wrapped_lines(text):
        lines = text.splitlines()
        out = []
        i, n = 0, len(lines)
        while i < n:
            buf = _TAG_RE.sub("", lines[i])
            j = i + 1
            while j < n:
                nxt = lines[j]
                stripped = nxt.strip()
                if not stripped:
                    break
                indent = len(nxt) - len(nxt.lstrip(" "))
                if indent < 15 or re.match(r"^\d{2}/\d{2}|^(INFO|WARNING|ERROR|DEBUG)\b", stripped):
                    break
                buf += " " + stripped
                j += 1
            out.append(buf)
            i = j
        return "\n".join(out)

    SUITE_NAMES = ["libero_10", "libero_goal", "libero_object", "libero_spatial"]
    suite_results = {}
    for suite in SUITE_NAMES:
        suite_dir = os.path.join(RESULTS_DIR, suite)
        wall_s = None
        wall_s_path = os.path.join(suite_dir, "wall_s.txt")
        if os.path.isfile(wall_s_path):
            try:
                wall_s = int(open(wall_s_path).read().strip())
            except Exception:
                wall_s = None

        exit_code = None
        exit_code_path = os.path.join(suite_dir, "exit_code.txt")
        if os.path.isfile(exit_code_path):
            exit_code = open(exit_code_path).read().strip()

        success_count, total_episodes = None, None
        eval_log_path = os.path.join(suite_dir, "eval.log")
        if os.path.isfile(eval_log_path):
            with open(eval_log_path, encoding="utf-8", errors="replace") as f:
                raw_text = f.read()
            reconstructed_text = reconstruct_wrapped_lines(raw_text)

            succ_matches = re.findall(r"#\s*successes:\s*(\d+)\s*\(", reconstructed_text)
            if succ_matches:
                success_count = int(succ_matches[-1])

            ep_matches = re.findall(r"Total episodes:\s*(\d+)", reconstructed_text)
            if ep_matches:
                total_episodes = int(ep_matches[-1])
            else:
                ep_fallback = re.findall(r"#\s*episodes completed so far:\s*(\d+)", reconstructed_text)
                if ep_fallback:
                    total_episodes = int(ep_fallback[-1])

        suite_results[suite] = {
            "wall_s": wall_s, "exit_code": exit_code,
            "success_count": success_count, "total_episodes": total_episodes,
        }

    suite_report_lines = []
    total_episodes_all, total_success_all, total_wall_s_all = 0, 0, 0
    suite_srs = []
    for suite in SUITE_NAMES:
        r = suite_results[suite]
        ep, succ, wall_s, exit_code = r["total_episodes"], r["success_count"], r["wall_s"], r["exit_code"]
        if ep and succ is not None:
            sr = succ / ep * 100
            suite_srs.append(sr)
            s_per_ep = (wall_s / ep) if wall_s else None
            total_episodes_all += ep
            total_success_all += succ
            total_wall_s_all += wall_s or 0
            suite_report_lines.append(
                f"[suite]  {suite:<14} ep={ep}  success={succ}  SR={sr:.1f}%  "
                f"wall_s={wall_s if wall_s is not None else 'NOT FOUND'}  "
                f"s_per_ep={f'{s_per_ep:.1f}' if s_per_ep else 'NOT FOUND'}"
            )
        else:
            suite_report_lines.append(
                f"[suite]  {suite:<14} ep=NOT FOUND  success=NOT FOUND  SR=NOT FOUND  "
                f"wall_s={wall_s if wall_s is not None else 'NOT FOUND'}  "
                f"s_per_ep=NOT FOUND  (exit_code={exit_code})"
            )

    mean_sr = (total_success_all / total_episodes_all * 100) if total_episodes_all else None
    projected_400_ep_hours = (
        (total_wall_s_all / total_episodes_all * 400 / 3600) if total_episodes_all else None
    )

    # protocol reminder printed unconditionally -- constraint #1: 1 trial/task
    # here vs. the published 50 trials/task (2000 episodes); nothing below
    # compares against published numbers.
    print("PROTOCOL: 1 trial/task (40 episodes total). Published LIBERO numbers use 50 trials/task (2000 episodes). Not comparable -- no comparison is made anywhere in this report.")

    if suite_srs and all(sr < 1.0 for sr in suite_srs):
        print("!" * 70)
        print("SR ~= 0 on EVERY suite. Constraint #3 (R3) says to try control_mode")
        print("reversed before suspecting the environment -- but no such parameter")
        print("exists anywhere in eval_libero.py, model2libero_interface.py, or Args")
        print("(searched the whole tree), and neither file may be edited (constraint #4).")
        print("This is reported as-is, unresolved -- needs a decision from Chu thau,")
        print("not a silent guess at what R3 actually refers to.")
        print("!" * 70)

    STAGE_STATUS["S8"] = "OK"
except Exception:
    STAGE_STATUS["S8"] = "FAILED"
    TRACEBACKS["S8"] = traceback.format_exc()
    print(TRACEBACKS["S8"])
    suite_report_lines = []
    total_episodes_all, mean_sr, total_wall_s_all, projected_400_ep_hours = 0, None, 0, None

print()
print("S8 parsing done, suite_results:", suite_results if 'suite_results' in dir() else "NOT AVAILABLE")

In [ ]:
report_lines = []
report_lines.append("=== COPY FROM HERE ===")
report_lines.append(f"[env]    gpu={REPORT['s0_gpu']}  vram_total={REPORT['s0_vram_gb']}GB  commit={REPORT['s1_commit']}")
report_lines.append(f"[env]    env_policy_numpy={REPORT['s2_numpy_version']}   env_sim_numpy={REPORT['s3_numpy_version']}")
report_lines.append(f"[ckpt]   base_vlm_in_config={REPORT['s5_base_vlm_in_config']}   patched_to={REPORT['s5_patched_to']}")
report_lines.extend(suite_report_lines)
report_lines.append(
    f"[total]  episodes={total_episodes_all}  "
    f"mean_SR={f'{mean_sr:.1f}%' if mean_sr is not None else 'NOT FOUND'}  "
    f"total_wall_s={total_wall_s_all}"
)
report_lines.append(
    "[proj]   400_ep_hours=" + (f"{projected_400_ep_hours:.2f}" if projected_400_ep_hours is not None else "NOT FOUND")
)
for stage in ["S0", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8"]:
    report_lines.append(f"[status] {stage}: {STAGE_STATUS.get(stage, 'NOT RUN')}")
report_lines.append("=== COPY TO HERE ===")

report_block = "\n".join(report_lines)
print(report_block)

with open("/content/libero_report.txt", "w", newline="\n") as f:
    f.write(report_block + "\n")

print()
print("Report also written to /content/libero_report.txt")

# TIP-009d bug #2: a stage that failed once and later succeeded on a rerun
# (same kernel) must not have its stale traceback printed again -- only
# show tracebacks for stages whose CURRENT status is FAILED.
active_tracebacks = {k: tb for k, tb in TRACEBACKS.items() if STAGE_STATUS.get(k) == "FAILED"}
if active_tracebacks:
    print()
    print("=== Full tracebacks (failed stages) ===")
    for key, tb in active_tracebacks.items():
        print(f"--- {key} ---")
        print(tb)
else:
    print()
    print("No tracebacks for currently-failed stages -- every stage that ran passed (any stale traceback here belongs to a stage that failed earlier and has since succeeded on a rerun).")